# Wo verliert der Nahverkehr gegen das Auto?

**Starter-Notebook für Projekt 1 in Informatik und Nachhaltigkeit (WiSe 2026/27)**

Dieses Notebook fragt den Routenplaner [MOTIS](https://github.com/motis-project/motis) ab,
der auf Ihrem Rechner läuft: erst eine einzelne Verbindung mit Bus und Bahn, mit dem Auto
und mit dem Rad, dann viele Startpunkte in Dortmund. Am Ende steht dieselbe Karte wie in
der [Webdemo](https://ls11ae.github.io/iun-routenplanung/reisezeit.html), nur gröber.

**Vorher:**
1. Datenpaket holen: `python daten/herunterladen.py`
2. MOTIS starten: im Ordner `nahverkehr` die Datei `starten.bat` doppelklicken (Windows) oder
   `./starten.sh` ausführen (macOS, Linux). Beim ersten Mal lädt das Skript MOTIS (52 MB) und
   bereitet die Daten vor (etwa eine Minute, gut 1 GB Arbeitsspeicher). Das Fenster offen lassen.
   Unter <http://localhost:8080> hat MOTIS auch eine eigene Oberfläche.

**Daten:** Fahrplan von [gtfs.de](https://gtfs.de) (DELFI e.V., CC BY 4.0), Wege aus
OpenStreetMap (© OpenStreetMap-Mitwirkende, ODbL).

In [1]:
import json
import statistics
import warnings
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

import pandas as pd

import reisezeitkarte as rk

try:
    print("Verbunden mit", rk.server_pruefen())
except Exception as fehler:
    raise SystemExit("MOTIS läuft nicht. Bitte zuerst nahverkehr/starten.bat (oder ./starten.sh) starten.") from fehler

Verbunden mit MOTIS 2.11.4


## 1. Gilt der Fahrplan noch?

Der Feed von gtfs.de deckt nur etwa einen Monat ab. Gerechnet wird für den nächsten
Dienstag, der im Fahrplan liegt. Zeiten gibt man MOTIS in Weltzeit (UTC); die Funktionen
in `reisezeitkarte.py` rechnen aus der Ortszeit um.

In [2]:
stand = json.loads((rk.WURZEL / "daten" / "stand.json").read_text(encoding="utf-8"))["gtfs"]
von, bis = (datetime.strptime(stand[k], "%Y%m%d").date() for k in ("gueltig_von", "gueltig_bis"))
print(f"Fahrplan gültig vom {von:%d.%m.%Y} bis {bis:%d.%m.%Y}.")
if date.today() > bis:
    warnings.warn("Der Fahrplan ist abgelaufen: git pull, python daten/herunterladen.py, dann MOTIS neu starten.")

beginn = max(von, date.today())
TAG = beginn + timedelta(days=(1 - beginn.weekday()) % 7)          # 1 = Dienstag
if TAG > bis:
    TAG = bis - timedelta(days=(bis.weekday() - 1) % 7)
ABFAHRT = rk.zeitpunkt(TAG.isoformat(), "07:30")                   # Ortszeit
wochentag = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"][TAG.weekday()]
print(f"Start: {wochentag}, {ABFAHRT:%d.%m.%Y, %H:%M} Uhr | in UTC: {rk.als_utc(ABFAHRT)}")

Fahrplan gültig vom 07.11.2026 bis 07.12.2026.
Start: Dienstag, 17.11.2026, 07:30 Uhr | in UTC: 2026-11-17T06:30:00Z


## 2. Eine Anfrage

Von Dortmund-Hörde zum Campus Nord. `rk.verbindungen` ruft die Schnittstelle
`/api/v6/plan` auf und liefert die Verbindungen mit ihren Abschnitten (*legs*).

In [3]:
START = (51.4862, 7.5050)      # Dortmund-Hörde (Breite, Länge)
ZIEL = rk.ZIEL                 # TU Dortmund, Campus Nord

ortszeit = lambda text: datetime.fromisoformat(text.replace("Z", "+00:00")).astimezone(ZoneInfo("Europe/Berlin"))

def als_tabelle(verbindungen):
    return pd.DataFrame([{
        "ab": ortszeit(v["startTime"]).strftime("%H:%M"), "an": ortszeit(v["endTime"]).strftime("%H:%M"),
        "Dauer (min)": round(v["duration"] / 60), "Umstiege": v["transfers"],
        "Weg": " → ".join(l.get("routeShortName") or ("zu Fuß" if l["mode"] == "WALK" else l["mode"]) for l in v["legs"]),
    } for v in verbindungen])

verbindungen = rk.verbindungen(START, ZIEL, ABFAHRT, timetableView="false")
als_tabelle(verbindungen)

,ab,an,Dauer (min),Umstiege,Weg
0,07:33,08:09,36,0,zu Fuß → 445 → zu Fuß


Und im Vergleich: `rk.oepnv_zeiten` fasst die Verbindungen zu einer Reisezeit zusammen
(früheste Ankunft minus Startzeit, die Wartezeit zählt also mit), `rk.strassenzeiten`
liefert Zeiten für Auto, Rad und Fußweg. Der Campus ist autofrei; das Auto fährt deshalb
bis zur Emil-Figge-Straße (`rk.ZIEL_AUTO`).

In [4]:
oepnv = rk.oepnv_zeiten(START, ZIEL, ABFAHRT)
pd.Series({
    "Bus und Bahn, Start 7:30 (min)": oepnv["min"],
    "… davon Fußweg zum ersten Halt (min)": oepnv["zum_halt_min"],
    "Bus und Bahn, schnellste Verbindung ohne Warten (min)": oepnv["dauer_min"],
    "Auto, reine Fahrzeit (min)": rk.strassenzeiten([START], rk.ZIEL_AUTO, "CAR")[0],
    "Fahrrad (min)": rk.strassenzeiten([START], ZIEL, "BIKE")[0],
    "zu Fuß (min)": rk.strassenzeiten([START], ZIEL, "WALK")[0],
}, name="Hörde → Campus Nord").to_frame()

,Hörde → Campus Nord
"Bus und Bahn, Start 7:30 (min)",39.0
… davon Fußweg zum ersten Halt (min),8.0
"Bus und Bahn, schnellste Verbindung ohne Warten (min)",33.0
"Auto, reine Fahrzeit (min)",10.8
Fahrrad (min),32.4
zu Fuß (min),106.4


## 3. Die Verbindung auf der Karte

Mit `detailedLegs="true"` enthält jeder Abschnitt seinen Verlauf als kodierte Linie
(*polyline*). Fußwege sind gepunktet, Fahrten durchgezogen gezeichnet.

In [5]:
import folium

def linie_dekodieren(text, genauigkeit=6):
    """Kodierte Linie (Google-Polyline-Verfahren) in eine Liste (Breite, Länge)."""
    punkte, i, lat, lon = [], 0, 0, 0
    while i < len(text):
        for achse in (0, 1):
            ergebnis, verschiebung = 0, 0
            while True:
                b = ord(text[i]) - 63
                i += 1
                ergebnis |= (b & 0x1F) << verschiebung
                verschiebung += 5
                if b < 0x20:
                    break
            d = ~(ergebnis >> 1) if ergebnis & 1 else ergebnis >> 1
            if achse == 0:
                lat += d
            else:
                lon += d
        punkte.append((lat / 10 ** genauigkeit, lon / 10 ** genauigkeit))
    return punkte

beste = min(rk.verbindungen(START, ZIEL, ABFAHRT, timetableView="false", detailedLegs="true"), key=lambda v: v["endTime"])
karte = folium.Map(location=START, zoom_start=13, tiles="OpenStreetMap")
for l in beste["legs"]:
    g = l["legGeometry"]
    zu_fuss = l["mode"] == "WALK"
    folium.PolyLine(linie_dekodieren(g["points"], g.get("precision", 6)), color="#555555" if zu_fuss else "#0072B2",
                    weight=4 if zu_fuss else 6, dash_array="1 8" if zu_fuss else None,
                    tooltip="zu Fuß" if zu_fuss else l.get("routeShortName", l["mode"])).add_to(karte)
folium.Marker(START, tooltip="Start").add_to(karte)
folium.Marker(ZIEL, tooltip="Ziel", icon=folium.Icon(color="black", icon="flag")).add_to(karte)
karte

## 4. Viele Startpunkte

`rk.raster_erzeugen` legt ein Gitter über das Stadtgebiet; `rk.reisezeiten_berechnen`
fragt für jeden Punkt Bus und Bahn, Auto, Rad und Fußweg ab. Hier mit 1000 m Maschenweite,
damit es schnell geht (etwa eine Minute). Die Karte der Webdemo hat 500 m:
`python nahverkehr/reisezeitkarte.py` (etwa vier Minuten).

In [6]:
RASTER = 1000
punkte = rk.raster_erzeugen(rk.grenze_laden(), RASTER)
zellen = rk.reisezeiten_berechnen(punkte, abfahrt=ABFAHRT)
t = pd.DataFrame(zellen)
t.describe().loc[["count", "min", "50%", "max"]].round(1)

Zu Fuß, Rad, Auto: 283 Punkte in 6.4 s


  ÖPNV:   100 von 283 Punkten,    15 s


  ÖPNV:   200 von 283 Punkten,    33 s


  ÖPNV:   283 von 283 Punkten,    49 s


ÖPNV: 283 Punkte in 49 s (12 Anfragen je Sekunde), Fehler: 0


,lat,lon,oepnv_min,oepnv_dauer_min,umstiege,zum_halt_min,auto_min,rad_min,fuss_min
count,283.0,283.0,280.0,280.0,280.0,276.0,273.0,280.0,280.0
min,51.4,7.3,7.8,7.8,-1.0,1.0,1.3,2.5,7.8
50%,51.5,7.5,52.0,42.5,1.0,13.0,12.6,39.5,132.4
max,51.6,7.6,95.0,77.0,3.0,56.0,24.1,74.7,252.9


## 5. Das Verhältnis und die Karte

Wie viel länger dauert es mit Bus und Bahn als mit dem Auto? Die Autozeit aus
OpenStreetMap ist ohne Stau und ohne Parkplatzsuche gerechnet. `ZUSCHLAG` sind Minuten
für Parken und den Fußweg vom Parkplatz: Probieren Sie 0, 5 und 10.

In [7]:
from abbildung_reisezeit import GRENZEN, NAMEN, RAMPE      # dieselben Klassen und Farben wie in der Abbildung

ZUSCHLAG = 5
t["verhaeltnis"] = t.oepnv_min / (t.auto_min + ZUSCHLAG)
t["klasse"] = t.verhaeltnis.apply(lambda v: None if pd.isna(v) else sum(v > g for g in GRENZEN))

dlon, dlat = rk.zelle_in_grad(ZIEL, RASTER)
karte = folium.Map(location=(51.515, 7.465), zoom_start=11, tiles="OpenStreetMap")
for z in t.itertuples():
    ecken = [(z.lat - dlat / 2, z.lon - dlon / 2), (z.lat + dlat / 2, z.lon + dlon / 2)]
    if pd.isna(z.verhaeltnis):
        folium.Rectangle(ecken, color="#555555", weight=1, dash_array="3 3", fill=True, fill_color="#8c8c88", fill_opacity=0.3, tooltip="kein Wert").add_to(karte)
    else:
        folium.Rectangle(ecken, color="#ffffff", weight=0.5, fill=True, fill_color=RAMPE[int(z.klasse)], fill_opacity=0.8,
                         tooltip=f"Bus und Bahn {z.oepnv_min:.0f} min, Auto {z.auto_min + ZUSCHLAG:.0f} min: {z.verhaeltnis:.1f}-mal so lange").add_to(karte)
folium.Marker(ZIEL, tooltip="Campus Nord", icon=folium.Icon(color="black", icon="flag")).add_to(karte)
karte

In [8]:
# Die Karte als Tabelle: Anteil der Zellen je Klasse, für verschiedene Zuschläge
def anteile(zuschlag):
    v = (t.oepnv_min / (t.auto_min + zuschlag)).dropna()
    klassen = v.apply(lambda x: sum(x > g for g in GRENZEN)).value_counts(normalize=True)
    return {**{NAMEN[k] + " so lange": klassen.get(k, 0.0) for k in range(len(NAMEN))}, "Median": v.median()}

pd.DataFrame({f"Auto + {z} min": anteile(z) for z in (0, 5, 10)}).style.format("{:.0%}").format("{:.1f}", subset=pd.IndexSlice[["Median"], :])

,Auto + 0 min,Auto + 5 min,Auto + 10 min
höchstens 2-mal so lange,0%,4%,26%
2- bis 3-mal so lange,8%,48%,67%
3- bis 4-mal so lange,28%,44%,6%
4- bis 5-mal so lange,36%,3%,1%
mehr als 5-mal so lange,28%,1%,0%
Median,4.3,3.0,2.3


**Lesehilfe:** Dunkle Zellen heißen nicht „schlechter Nahverkehr“, sondern „großes
Verhältnis“. Nah am Ziel ist das Auto in zwei Minuten da; zu Fuß oder mit dem Bus dauert
es dann schnell fünfmal so lange, obwohl es nur um wenige Minuten geht. Mit einem Zuschlag
für das Parken verschwindet dieser Effekt weitgehend. Das Raster ist außerdem gleichmäßig
über die Fläche gelegt: Felder und Industriegebiete zählen so viel wie Wohnviertel.

## 6. Ihr Projekt: Ideen

1. **Annahmen prüfen und begründen:** Parkplatzsuche und Stau (Zuschlag je Stadtteil oder
   Tageszeit?), Startzeit und Takt (früheste Ankunft oder schnellste Verbindung?),
   Fußweg zum Halt (hier bis 60 Minuten erlaubt), Verspätungen.
2. **Für wen?** Gewichten Sie die Zellen mit Einwohnerzahlen (Zensus 2022, 100-m-Raster).
   Und: Wer hat überhaupt ein Auto?
3. **Andere Ziele und Zeiten:** Innenstadt, Klinikum, abends, am Sonntag. Wie stabil ist das Bild?
4. **Das Rad:** als eigenes Verkehrsmittel (`rad_min`) und als Zubringer zur S-Bahn.
5. **Wo ist der Engpass, und was ändert eine einzelne Maßnahme?** Dafür ist es praktisch,
   den Fahrplan selbst in der Hand zu haben: siehe `fahrplan_selbst_rechnen.ipynb`.
6. **Barrierefrei:** MOTIS kennt ein Rollstuhlprofil (`pedestrianProfile="WHEELCHAIR"` als
   weiterer Parameter von `rk.verbindungen`). Wie ändert sich die Karte?

Bitte nennen Sie auf Ihrem Poster die Quellen: Fahrplan gtfs.de / DELFI e.V. (CC BY 4.0),
Karte © OpenStreetMap-Mitwirkende (ODbL), berechnet mit MOTIS.